<a href="https://colab.research.google.com/github/zynpozkk/BTK_Python_Egitimi/blob/main/kayseri_travel_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pdfplumber langchain langchain-text-splitters \
                sentence-transformers chromadb groq gradio
print("✅ Kütüphaneler kuruldu")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [2]:
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq API bağlandı")

✅ Groq API bağlandı


In [3]:
EMBEDDING_MODEL = "distiluse-base-multilingual-cased-v1"
GROQ_MODEL      = "llama-3.3-70b-versatile"
COLLECTION_NAME = "kayseri_travel"
CHUNK_SIZE      = 600
CHUNK_OVERLAP   = 100
TOP_K           = 5

SYSTEM_PROMPT = """Sen Kayseri şehrini gezen turistlere yardım eden uzman bir seyahat rehberisin.
Sana verilen BAĞLAM bilgilerine dayanarak soruları Türkçe, sıcak ve net bir dille yanıtla.

KURALLAR:
- Sadece BAĞLAM'daki bilgilere dayan, asla uydurma yapma.
- Bağlamda bilgi yoksa "Bu konuda elimde net bilgi yok, ancak..." de.
- Fiyat, adres, telefon gibi spesifik detayları VARSA mutlaka belirt.
- Cevaplar samimi olsun; turist gerçekten yanında bir rehber varmış gibi hissetsin.
- Gerektiğinde madde işaretiyle düzenli sun.
- Soruya doğrudan cevap ver, gereksiz uzatma yapma.
"""
print("✅ Ayarlar tamam")

✅ Ayarlar tamam


In [4]:
from google.colab import files
import pdfplumber

print("📄 PDF dosyasını yükleyin...")
pdf_upload = files.upload()
PDF_PATH = list(pdf_upload.keys())[0]
print(f"✅ PDF yüklendi: {PDF_PATH}")

📄 PDF dosyasını yükleyin...


Saving kayseri_dijital_rehber.pdf to kayseri_dijital_rehber.pdf
✅ PDF yüklendi: kayseri_dijital_rehber.pdf


In [5]:
def load_pdf(path):
    pages = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({"page_num": i, "content": text})
    return pages

pages = load_pdf(PDF_PATH)
print(f"✅ {len(pages)} sayfa okundu")
print(f"\n--- Örnek ---\n{pages[1]['content'][:200]}")

✅ 19 sayfa okundu

--- Örnek ---
1. GENEL BİLGİLER VE ŞEHİR PROFİLİ
Kayseri, İç Anadolu Bölgesi'nin doğu kapısı ve Kapadokya'ya en yakın büyük şehir. Erciyes
Dağı'nın (3.916 m) eteklerine kurulu olan şehir, Hitit (Kültepe/Kaniş), Rom


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len
)

chunks = []
for page in pages:
    page_chunks = splitter.split_text(page["content"])
    for chunk in page_chunks:
        if len(chunk.strip()) > 50:
            chunks.append({"text": chunk, "page": page["page_num"]})

print(f"✅ {len(chunks)} chunk oluşturuldu")
print(f"Ortalama uzunluk: {sum(len(c['text']) for c in chunks)//len(chunks)} karakter")

✅ 53 chunk oluşturuldu
Ortalama uzunluk: 484 karakter


In [7]:
from sentence_transformers import SentenceTransformer

print("⏳ Model yükleniyor (1-2 dk sürebilir)...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model hazır!")

⏳ Model yükleniyor (1-2 dk sürebilir)...


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

✅ Model hazır!


In [8]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(Settings(anonymized_telemetry=False))

try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass

collection = client.create_collection(name=COLLECTION_NAME)

print("⏳ Chunk'lar vektöre dönüştürülüyor...")
texts = [c["text"] for c in chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"page": c["page"], "chunk_id": i} for i, c in enumerate(chunks)],
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ {len(chunks)} chunk kaydedildi!")

⏳ Chunk'lar vektöre dönüştürülüyor...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ 53 chunk kaydedildi!


In [9]:
def retrieve(query, top_k=TOP_K):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return [{
        "text": doc,
        "page": meta["page"],
        "distance": dist
    } for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )]

print("✅ retrieve() hazır")

✅ retrieve() hazır


In [10]:
def chat(query, show_sources=False):
    retrieved = retrieve(query)

    context = "\n\n---\n\n".join([
        f"[Sayfa {r['page']}]\n{r['text']}"
        for r in retrieved
    ])

    full_prompt = f"""BAĞLAM (Kayseri rehberinden):
{context}

KULLANICI SORUSU: {query}

Yukarıdaki bağlama dayanarak soruyu Türkçe cevapla."""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": full_prompt}
        ],
        max_tokens=1024,
        temperature=0.3,
    )

    answer = response.choices[0].message.content

    if show_sources:
        answer += f"\n\n📚 Kaynak sayfalar: {sorted(set(r['page'] for r in retrieved))}"

    return answer

print("✅ chat() hazır")
print("\n🧪 Test ediliyor...")
print(chat("Kayseri'de ne yenir?"))

✅ chat() hazır

🧪 Test ediliyor...
Kayseri mutfağı gerçekten çok zengin! İki temel üzerine kurulu: ETten gelen ürünler ve hamur işleri. Mutlaka tadına bakmanız gerekenler:
* Kayseri Mantısı: 40 tanesi kaşığa sığacak küçüklükte, yoğurt, tereyağı ve pul biber ile servis edilir.
* Tepsi Mantısı: Fırın versiyonu, kıyma harçlı ve tepside hazırlanır.
* Yağlama: İnce yufka arasından kıyma soslu katmanlı börek.
* Pastırma ve sucuk: Coğrafi işaret tescillidir ve Kayseri dışında yapılanlara 'Kayseri pastırması' denmez.
* Bonfile, kontrafile ve kuşgömü gibi pastırma çeşitleri de denenmeli.
* Ayrıca, Kayseri baklavası veya nevzine tatlısı da lezzetli seçeneklerden. 

Bu lezzetleri denediğinizde, Kayseri mutfağının güzelliğini keşfedeceksiniz!


In [11]:
from google.colab import files
from PIL import Image
import base64, io

# Erciyes
print("📷 Erciyes resmini yükleyin...")
erc = files.upload()
erc_name = list(erc.keys())[0]
img = Image.open(erc_name)
img.thumbnail((1200, 600))
buf = io.BytesIO()
img.save(buf, format="JPEG", quality=85)
ERCIYES_URL = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()
print(f"✅ Erciyes hazır ({len(ERCIYES_URL)//1024} KB)\n")

# Vosvos
print("📷 Vosvos resmini yükleyin...")
vos = files.upload()
vos_name = list(vos.keys())[0]
img2 = Image.open(vos_name)
img2.thumbnail((800, 600))
buf2 = io.BytesIO()
img2.save(buf2, format="PNG", optimize=True)
VOSVOS_URL = "data:image/png;base64," + base64.b64encode(buf2.getvalue()).decode()
print(f"✅ Vosvos hazır ({len(VOSVOS_URL)//1024} KB)")

📷 Erciyes resmini yükleyin...


Saving erciyes.jpg to erciyes.jpg
✅ Erciyes hazır (134 KB)

📷 Vosvos resmini yükleyin...


Saving vosvos.png to vosvos.png
✅ Vosvos hazır (616 KB)


In [15]:
import gradio as gr

def chat_for_ui(message, history):
    try:
        show_src = message.startswith("?")
        if show_src:
            message = message[1:].strip()
        return chat(message, show_sources=show_src)
    except Exception as e:
        return f"❌ Hata: {str(e)}"

KARSILAMA = """🏛️ **Merhaba! Ben Caesarea AI.**

Kayseri'nin dijital seyahat rehberiyim. Roma döneminde **Caesarea Mazaca** olarak bilinen bu kadim şehri keşfetmene yardımcı olacağım!

🍽️ Yeme-içme · 🏛️ Tarihi yerler · 🚌 Ulaşım
⛷️ Erciyes kayak · 🗺️ Günlük rotalar · 🧳 Konaklama"""

KAYSERI_SVG = """
<div style="width:100%; overflow:hidden; height:130px; margin-bottom:8px;">
<svg viewBox="0 0 680 130" xmlns="http://www.w3.org/2000/svg"
     style="width:100%; height:130px;">
  <defs>
    <style>
      .dl1{stroke-dasharray:4000;stroke-dashoffset:4000;animation:draw 3.5s ease forwards;}
      .dl2{stroke-dasharray:2000;stroke-dashoffset:2000;animation:draw 3s ease 0.8s forwards;}
      .dl3{stroke-dasharray:1000;stroke-dashoffset:1000;animation:draw 2s ease 1.2s forwards;}
      .tf{opacity:0;animation:fadeIn 1.5s ease 3s forwards;}
      @keyframes draw{to{stroke-dashoffset:0;}}
      @keyframes fadeIn{to{opacity:1;}}
    </style>
  </defs>

  <!-- Erciyes arka planda ince -->
  <path class="dl3"
    d="M240 90 L268 55 L280 65 L294 45 L310 55 L322 35 L334 18 L340 8 L346 18 L358 35 L370 55 L384 45 L398 65 L410 55 L438 90"
    stroke="#D4A853" stroke-width="1.8" fill="none"
    stroke-linecap="round" stroke-linejoin="round" opacity="0.55"/>

  <!-- Kale ana çizgisi -->
  <path class="dl1"
    d="M30 105
       L30 88 L38 88 L38 82 L46 82 L46 88 L54 88 L54 82 L62 82 L62 88 L70 88 L70 105
       L80 105 L80 92 L95 92 L95 82 L103 82 L103 75 L111 75 L111 82 L119 82 L119 75 L127 75 L127 82 L135 82 L135 92 L148 92
       L148 72 L156 72 L156 65 L164 65 L164 72 L172 72 L172 65 L180 65 L180 72 L188 72 L188 105
       L200 105 L200 88 L212 88 L212 80 L220 80 L220 73 L228 73 L228 80 L236 80 L236 73 L244 73 L244 80 L252 80 L252 88 L262 88
       L262 62 L270 62 L270 55 L278 55 L278 62 L286 62 L286 55 L294 55 L294 62 L302 62 L302 55 L310 55 L310 62 L318 62 L318 105
       L328 105 L328 85 L340 85 L340 78 L348 78 L348 85 L360 85
       L360 62 L368 62 L368 55 L376 55 L376 62 L384 62 L384 55 L392 55 L392 62 L400 62 L400 105
       L410 105 L410 88 L422 88 L422 80 L430 80 L430 72 L438 72 L438 80 L446 80 L446 88 L458 88
       L458 70 L466 70 L466 63 L474 63 L474 70 L482 70 L482 63 L490 63 L490 70 L498 70 L498 105
       L510 105 L510 90 L522 90 L522 82 L530 82 L530 90 L542 90 L542 82 L550 82 L550 90 L562 90
       L562 105 L572 105 L572 95 L582 95 L582 88 L592 88 L592 95 L602 95 L602 105
       L640 105"
    stroke="#F5F0E8" stroke-width="2" fill="none"
    stroke-linecap="round" stroke-linejoin="round"/>

  <!-- Kapı kemerleri -->
  <path class="dl2"
    d="M278 105 L278 82 Q294 70 310 82 L310 105
       M376 105 L376 82 Q390 72 404 82"
    stroke="#F5F0E8" stroke-width="1.8" fill="none"
    stroke-linecap="round" stroke-linejoin="round"/>

  <!-- Zemin -->
  <path class="dl1"
    d="M20 105 L650 105"
    stroke="#D4A853" stroke-width="1.5" fill="none" opacity="0.6"/>

  <!-- Kayseri imza -->
  <path class="dl2"
    d="M230 122 Q248 113 262 120 Q275 127 285 120 Q297 111 308 118 Q322 126 334 119 Q347 110 358 118 Q372 126 384 119 Q396 110 410 118 Q422 126 438 118"
    stroke="#D4A853" stroke-width="2" fill="none"
    stroke-linecap="round" stroke-linejoin="round"/>
</svg>
</div>
"""

ornek_sorular = [
    ["🍽️ Kayseri'de ne yenir?"],
    ["🗺️ 1 günde Kayseri'yi nasıl gezerim?"],
    ["⛷️ Erciyes'te kayak fiyatları nedir?"],
    ["✈️ Havalimanından şehre nasıl giderim?"],
    ["🥩 Pastırma için en iyi yer neresi?"],
    ["👨‍👩‍👧 Çocuklarla nereye gidebilirim?"],
    ["🏔️ Kapadokya'ya nasıl giderim?"],
    ["🏛️ Mimar Sinan ile ilgili yer var mı?"],
]

custom_css = f"""
/* Ana arka plan — koyu bordo */
.gradio-container {{
    background: linear-gradient(160deg, #2C0A0A 0%, #4A1010 40%, #3D0E0E 70%, #2A0808 100%) !important;
    min-height: 100vh;
    font-family: 'Segoe UI', sans-serif !important;
}}

/* Sol sidebar */
#sidebar {{
    background: rgba(0,0,0,0.35) !important;
    border-right: 1px solid rgba(212,168,83,0.3) !important;
    padding: 20px 15px !important;
    min-height: 100vh !important;
}}

/* Chatbot alanı */
.gradio-container [data-testid="chatbot"] {{
    background: rgba(20,5,5,0.85) !important;
    border: 1px solid rgba(212,168,83,0.25) !important;
    border-radius: 14px !important;
}}

/* Bot mesajı */
.gradio-container [data-testid="bot"] {{
    background: rgba(60,15,15,0.9) !important;
    border: 1px solid rgba(212,168,83,0.2) !important;
    color: #F5F0E8 !important;
    border-radius: 12px !important;
}}

/* Kullanıcı mesajı */
.gradio-container [data-testid="user"] {{
    background: linear-gradient(135deg, #7B1C1C, #9B2525) !important;
    color: #F5F0E8 !important;
    border-radius: 12px !important;
}}

/* Input kutusu */
.gradio-container textarea,
.gradio-container input[type="text"] {{
    background: rgba(20,5,5,0.85) !important;
    color: #F5F0E8 !important;
    border: 1px solid rgba(212,168,83,0.35) !important;
    border-radius: 12px !important;
    font-size: 14px !important;
}}
.gradio-container textarea::placeholder {{
    color: #8B6B6B !important;
}}

/* Gönder butonu */
.gradio-container button.primary {{
    background: linear-gradient(135deg, #D4A853, #B8860B) !important;
    color: #1A0505 !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 10px !important;
}}

/* Temizle butonu */
.gradio-container button.secondary {{
    background: rgba(255,255,255,0.06) !important;
    color: #C4A882 !important;
    border: 1px solid rgba(212,168,83,0.2) !important;
    border-radius: 10px !important;
}}

/* Örnek sorular */
.gradio-container .examples button {{
    background: rgba(212,168,83,0.1) !important;
    color: #D4A853 !important;
    border: 1px solid rgba(212,168,83,0.3) !important;
    border-radius: 20px !important;
    font-size: 12px !important;
    padding: 6px 14px !important;
    margin: 3px !important;
}}
.gradio-container .examples button:hover {{
    background: rgba(212,168,83,0.22) !important;
}}

/* Label renkleri */
.gradio-container label,
.gradio-container .label-wrap span {{
    color: #D4A853 !important;
    font-weight: 600 !important;
}}

footer {{ display: none !important; }}
"""

with gr.Blocks(
    theme=gr.themes.Base(primary_hue="orange"),
    css=custom_css,
    title="Caesarea AI — Kayseri Travel Chatbot"
) as demo:

    with gr.Row():
        # ── SOL SIDEBAR ──────────────────────────────────
        with gr.Column(scale=1, min_width=200, elem_id="sidebar"):
            gr.HTML("""
            <div style="color:#D4A853; font-size:20px; font-weight:800;
                        text-align:center; margin-bottom:20px; letter-spacing:1px;">
                🏛️ Caesarea AI
            </div>
            """)

            new_chat_btn = gr.Button("+ Yeni Sohbet", variant="primary")

            gr.HTML("""
            <div style="color:#6B4B4B; font-size:11px;
                        margin-top:20px; text-align:center;">
                ──── Hakkında ────
            </div>
            <div style="color:#C4A882; font-size:11px;
                        margin-top:10px; line-height:2;">
                🤖 Groq LLaMA 3.3 70B<br>
                📚 ChromaDB + RAG<br>
                📄 Kayseri Rehberi PDF<br>
                🎓 BTK Akademi Projesi
            </div>
            """)

            gr.HTML(f"""
            <div style="margin-top:30px; text-align:center;">
                <img src="{VOSVOS_URL}"
                     style="max-width:100%; border-radius:8px; opacity:0.85;">
            </div>
            """)

        # ── SAĞ ANA ALAN ──────────────────────────────────
        with gr.Column(scale=4):

            # Animasyonlu SVG — ince şerit
            gr.HTML(KAYSERI_SVG)

            # Başlık
            gr.HTML("""
            <div style="text-align:center; margin: 0 0 12px;">
                <span style="font-size:26px; font-weight:900; color:#F5F0E8;
                             letter-spacing:2px;">🏛️ Caesarea AI</span><br>
                <span style="font-size:13px; color:#D4A853; letter-spacing:1px;">
                    Roma'dan günümüze Kayseri'nin dijital rehberi
                </span>
            </div>
            """)

            chatbot = gr.Chatbot(
                value=[{"role": "assistant", "content": KARSILAMA}],
                height=400,
                type='messages',
                avatar_images=(
                    None,
                    "https://cdn-icons-png.flaticon.com/512/4712/4712027.png"
                ),
                show_copy_button=True,
                label="💬 Sohbet"
            )

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Kayseri hakkında bir şey sorun...",
                    show_label=False,
                    scale=5,
                    container=False,
                    lines=1
                )
                send_btn = gr.Button("Gönder ➤", variant="primary", scale=1, min_width=100)
                clear_btn = gr.Button("🗑️ Temizle", scale=1, min_width=100)

            gr.Examples(
                examples=ornek_sorular,
                inputs=msg,
                label="💡 Örnek sorular"
            )

    def respond(message, chat_history):
        if not message.strip():
            return "", chat_history
        bot_message = chat_for_ui(message, chat_history)
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        return "", chat_history

    def yeni_sohbet():
        return [{"role": "assistant", "content": KARSILAMA}]

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(
        lambda: [{"role": "assistant", "content": KARSILAMA}],
        outputs=chatbot
    )
    new_chat_btn.click(yeni_sohbet, outputs=chatbot)

demo.launch(share=True, debug=False)

/tmp/ipykernel_3356/1421149054.py:182: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3356/1421149054.py:182: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3356/1421149054.py:238: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3356/1421149054.py:238: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77590d37ebca486ffc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
